In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user", 
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant", 
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [4]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [13]:
# first tool 

from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")

    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam( {
  "name": "get_current_datetime",
  "description": "Returns the current date and time on the server, formatted as a string according to the provided strftime-style format pattern. Use this tool whenever the user asks for the current date, current time, today's date, or needs a timestamp for the present moment — it should not be used to look up dates/times in other timezones, historical dates, or future dates, since it only returns the system's current local time. If no format is specified, it defaults to 'YYYY-MM-DD HH:MM:SS' (24-hour clock). The format parameter must be a valid Python strftime pattern (e.g. '%Y-%m-%d' for just the date, '%I:%M %p' for 12-hour time with AM/PM, '%A, %B %d, %Y' for a full weekday/month/day/year string); passing an empty string will cause an error rather than falling back to a default.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string controlling how the date/time is rendered (e.g. '%Y-%m-%d %H:%M:%S' for '2026-07-15 14:30:00', '%m/%d/%Y' for '07/15/2026', '%I:%M %p' for '02:30 PM'). Must not be an empty string, as that will raise an error. If omitted, defaults to '%Y-%m-%d %H:%M:%S'.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  },
  "input_examples": [
    {},
    {"date_format": "%Y-%m-%d"},
    {"date_format": "%I:%M %p"},
    {"date_format": "%A, %B %d, %Y"}
  ]
})

In [35]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formated as HH:MM:SS.sss"
    }
)

response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema],
)

# add both blocks (text and tool use) to the list of messages
messages.append({
    "role": "assistant",
    "content": response.content
})

In [36]:
print(response.content)

# get the requested input of the tool function call claude requested
tool_use_blocks = [b for b in response.content if b.type == "tool_use"] #gets all tool_use blocks

tool_use_block = tool_use_blocks[0]

# we expect only one tool call (we only have one tool right now)
print(tool_use_blocks[0].input)

# run the tool function with the input param claude generated
datetime_result = get_current_datetime(**tool_use_blocks[0].input)

print(tool_use_block.id)



[ToolUseBlock(id='toolu_01BSPdqZ4kWPweuGQF4eyp3A', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S.%f'}, name='get_current_datetime', type='tool_use')]
{'date_format': '%H:%M:%S.%f'}
toolu_01BSPdqZ4kWPweuGQF4eyp3A


In [37]:
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": datetime_result,
            "is_error": False
        }
    ]
})

messages

[{'role': 'user',
  'content': 'What is the exact time, formated as HH:MM:SS.sss'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01BSPdqZ4kWPweuGQF4eyp3A', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S.%f'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01BSPdqZ4kWPweuGQF4eyp3A',
    'content': '20:52:44.128096',
    'is_error': False}]}]

In [38]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_011Cd4pdYt544hoNoDpL52qX', container=None, content=[TextBlock(citations=None, text='The exact time is **20:52:44.128** (in HH:MM:SS.sss format, showing hours, minutes, seconds, and milliseconds).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1229, output_tokens=40, output_tokens_details=None, server_tool_use=None, service_tier='standard'))